# Quick Overview: Cross-Correlation and Likelihood Mapping


This section performs cross-correlation analysis to detect planetary signals, following methods outlined in 
Boucher et al. (2021, 2023) and Gibson et al. (2020)

Functions also include likelihood calculation, velocity models, and plotting utilities for correlation maps and significance testing. 

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Welcome to the workshop version of this notebook! This notebook will NOT fully run through. We have placed highlighted questions to help guide you with what needs to be fixed/changed. If needed feel free to take a peek at the answer key if you are stuck!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  all the answers/ hints will be shown in these drop down menus!
</details>

In [ ]:
# We are NOT running petitRADTRANS in this tutorial. However, Starships needs it to run. 
# We are tricking the container here to believe we have petitRADTRANS when in fact all the functions are empty

import sys
import types

# Top-level dummy package
petitRADTRANS = types.ModuleType("petitRADTRANS")

# Submodules as separate mock modules/namespaces
petitRADTRANS.Radtrans = object  # or define a dummy class

petitRADTRANS.nat_cst = types.SimpleNamespace()
petitRADTRANS.physics = types.SimpleNamespace(
    guillot_global=lambda *a, **k: None,
    guillot_modif=lambda *a, **k: None
)
petitRADTRANS._read_opacities = types.SimpleNamespace()
petitRADTRANS.fort_input = types.SimpleNamespace()
petitRADTRANS.fort_rebin = types.SimpleNamespace()
petitRADTRANS.pyth_input = types.SimpleNamespace()

poor_mans_nonequ_chem = types.ModuleType("poor_mans_nonequ_chem")
poor_mans_nonequ_chem.interpol_abundances = lambda *a, **k: None

# Register everything in sys.modules
sys.modules["petitRADTRANS"] = petitRADTRANS
sys.modules["petitRADTRANS.radtrans"] = petitRADTRANS
sys.modules["petitRADTRANS._read_opacities"] = petitRADTRANS._read_opacities
sys.modules["petitRADTRANS.fort_input"] = petitRADTRANS.fort_input
sys.modules["petitRADTRANS.fort_rebin"] = petitRADTRANS.fort_rebin
sys.modules["petitRADTRANS.pyth_input"] = petitRADTRANS.pyth_input
sys.modules["petitRADTRANS.nat_cst"] = petitRADTRANS.nat_cst
sys.modules["petitRADTRANS.physics"] = petitRADTRANS.physics
sys.modules["petitRADTRANS.poor_mans_nonequ_chem"] = poor_mans_nonequ_chem

In [ ]:
# === Set up plotting inline for Jupyter ===
%matplotlib inline

# === Standard Library Imports ===
import os
import logging
import warnings
from pathlib import Path
from sys import path
from itertools import product

# === Scientific Libraries ===
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const
from matplotlib import gridspec, cm, set_loglevel
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase
from astropy import stats

# === Starships Module Imports ===
import starships.plotting_fcts as pf
from starships import homemade as hm
import starships.correlation_class as cc
from starships.correlation_class import Correlations
import starships.correlation as corr
import starships.planet_obs as pl_obs
from starships.planet_obs import Observations, Planet

# === Miscellaneous Settings ===

# Suppress common warnings for cleaner output
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

# Suppress verbose fontTools logging output
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)

# === Load custom colors for plotting ===
couleurs = hm.get_colors('magma', 50)[5:-2]


# Load in the reduction files

This section loads the reduction files that were generated in the previous notebook.
We'll also define the planet parameters here — make sure they match the ones used in the earlier reduction step.

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Some of these parameters are wrong! Make sure they match your reduction files!
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  M_star, Period and inclination need to be fixed
</details>

In [ ]:

# Input planet name, this will pull from exofile, so we can change any parameters here
# link to the NASA Achive:https://exoplanetarchive.ipac.caltech.edu/overview/WASP%20127

# === Planetary and Stellar Parameters that can be changed===
# Make sure to set the same as the reduction notebook

pl_name = 'WASP-127 b'
planet_obj = Planet(pl_name)

# Stellar parameters
planet_obj.M_star = 1.950 * const.M_sun     # Mass of the star
planet_obj.R_star = 1.333 * u.R_sun         # Radius of the star
planet_obj.Teff = 5842 * u.K                # Effective temperature of the star

# Planetary parameters
planet_obj.M_pl = 0.165 * u.M_jup           # Mass of the planet
planet_obj.R_pl = 1.311 * u.R_jup           # Radius of the planet
planet_obj.Tp = 1400 * u.K                  # Planet temperature

# Orbital parameters
planet_obj.period = 6.178062 * u.day        # Orbital period
planet_obj.trandur = 0.181 * u.day          # Transit duration
planet_obj.ap = 0.04840 * u.au              # Semi-major axis
planet_obj.incl = 85.85 * u.deg             # Orbital inclination
planet_obj.excent = 0.0                     # Eccentricity
planet_obj.w = (-90 * u.deg).to(u.rad)      # Argument of periastron (converted to radians)

In [ ]:
#give a path to the files we reduced in the previous notebook, this is the same as your outdir in the previous notebook
reduc_dir = '/home/jovyan/Notebooks/WASP-127b_ExoSLAM_Workshop'


In [ ]:
#list the file names you would like to use here
filename_list = ['sequence_2-pc_mask_wings90_data_trs_WASP127b_TR1']
obs_list = []

#this will run through all of the name and create an "obs_list"(a list of our observations)

for filename in filename_list:
    obs = pl_obs.load_single_sequences(filename, pl_name, path=reduc_dir,
                              load_all=False, filename_end='', plot=False, planet=planet_obj)
    obs_list.append(obs)
    
#the obs list will have all of the information from the reduction files

# Next, we need to load in the model
The notebook "Make a Model" uses PetitRADTRANS (PRT) to generate a planetary atmosphere model and saves it to a NPZ file. PetitRADTRANS is a tool for simulating the transmission or emission spectra of exoplanet atmospheres, based on properties like temperature, composition, and pressure.

However, as we are not using PRT in this notebook, we aren't going to build the model from scratch. Instead, We have already created models for you to try! The models are located in a folder called "Models" in the container. Your should see four different models there. Below, we show you how to load in one and plot it.

The NPZ files are composed of a wavelength componant and the spectra

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
There are four different models in total. What do the other models look like? What is the difference between them? Which one should you use to cross correlate with?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

Model one is water without clouds, Model two is OH without clouds and Model three is water as an emission model. Model four is the Best fit model from a retrieval. This is the one you should cross-correlate with.
</details>

In [ ]:
model_file = np.load('/home/jovyan/Models/ FILL THIS IN ')
wave_mod, model_spec = model_file['wave'], model_file['spec']

plt.plot(wave_mod, model_spec)
plt.title('Models')

# Let's run the Cross Correlation

First we need to define a few extra parameters and then we are ready to run!

In [ ]:
#where do you want the proucts stored?
output_dir = '/home/jovyan/Notebooks/WASP-127b_ExoSLAM_Workshop/'

print(output_dir)

In [ ]:
corr.calc_logl_injred?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Fill in the missing parameters: kind_trans, visit name. All fill in the model you would like to use and the model name
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Kind trans is 'transmission' and visit_name should be similar to that in the reduction so something like "WASP127b_TR1" but you can pick! The model name should be something districptive to help you remember! 
</details>

In [ ]:
# === Define parameters for correlation ===

# Define the visit name for file naming
visit_name = 'FILL THIS IN '

# Define model type and type of data (transmission or emission)
kind_trans = 'FILL THIS IN '

# Define RV grid for injected signal
n_RV_inj = 61 #number of steps
corrRV0 = np.linspace(-30, 30, n_RV_inj)  # RVs for cross-correlation


# ===== Set up the model to cross correlate with ======


#Define the model you wish to use to cross correlated
model_file = np.load('/home/jovyan/Models/ FILL IN THE MODEL YOU WANT TO USE')
wave_mod, model_spec = model_file['wave_mod'], model_file['model_spec']

#model name that you are using this will go into naming the file
model_name = 'FILL THIS IN'


# File name for the reduction you would like to cross correlate [can be a list]
filename_list = [f'sequence_{n_pc}-pc_mask_wings{mask_w}_data_trs_WASP127b_TR1'
                 for mask_w, n_pc in product([90], range(2, 3))]
print('\n'.join(filename_list))


# # === Initialize storage for results ===


n_pc_list = []
mask_wings_list = []
all_obs = dict()
all_ccf_map = dict()
all_logl_map = dict()


# # === Loop through filelist to cross correlate if desired ===

for filename in filename_list:
    obs = pl_obs.load_single_sequences(filename, pl_name, path=reduc_dir,
                              load_all=False, filename_end='', plot=False, planet=planet_obj)
    
    # Generate Kp from the reduction file
    Kp_array = np.array([obs.Kp.value]) 
        
    n_pc = int(obs.params[5])
    n_pc_list.append(n_pc)
    print('n_pc_list',n_pc_list)
    
    mask_wings = int(obs.params[1] * 100)  # in percent
    mask_wings_list.append(mask_wings)
    print('mask_wings_list',mask_wings_list)

    out_filename = f'{Path(filename).stem}_ccf_logl_seq_{model_name}'

    ccf_map, logl_map = corr.calc_logl_injred(
        obs,'seq', planet_obj, Kp_array, corrRV0, [n_pc], wave_mod, model_spec,  kind_trans
    )


    corr.save_logl_seq(output_dir / Path(out_filename), ccf_map, logl_map,
                        wave_mod, model_spec, n_pc, Kp_array, corrRV0, kind_trans)


    all_obs[(n_pc, mask_wings)] = obs
    all_ccf_map[(n_pc, mask_wings)] = ccf_map
    all_logl_map[(n_pc, mask_wings)] = logl_map

# That's it! Let's look at the results!

This can also be done in a separate notebook

In [ ]:
cc.plot_ccflogl?

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
What happens to the detection if you changes the orders we are looking at?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  The detection might change based on the orders we use! For water since it stretches the whole model we will want to use all the order but form something like CO that is only in a small range, we only need specific orders
</details>

In [ ]:
# === Plot Single CCF and Log-Likelihood for a Given PCA and Masking Setup ===

# Select the PCA number and mask wings percentage to analyze
n_pc, mask_w = 2, 90

# Define the spectral orders to include in the cross-correlation function (CCF)
# You can specify a single order as an integer or multiple orders as a list/array

order_indices = np.arange(75)  # Example: all orders <75
# order_indices = [46, 47]     # Example for multiple orders (uncomment to use)

print(f"Orders used for CCF and log-likelihood calculation: {order_indices}")

# Define the filename for the reduction file to analyze
filename = f'sequence_{n_pc}-pc_mask_wings{mask_w}_data_trs_WASP127b_TR1'

# Load the observation data for the specified filename and planet
obs = pl_obs.load_single_sequences(
    filename, pl_name, path=reduc_dir,
    load_all=False, filename_end='', plot=False,
    planet=planet_obj
)

# Retrieve the relevant precomputed objects from dictionaries by (n_pc, mask_wings) key from the above cell
args = [all_something[(n_pc, mask_wings)] for all_something in [all_obs, all_ccf_map, all_logl_map]]

# Compute and plot the CCF and log-likelihood objects for the specified orders
# 'plot_ccflogl' returns ccf and logl objects, and optionally plots figures
ccf_obj, logl_obj = cc.plot_ccflogl(
    *args, #pulling in the save directories from above
    corrRV0, Kp_array, [n_pc], #defined above
    RV= -8, # central value to compute significance
    orders=order_indices, #defined above
    path_fig=" ",    # Path to save figures (empty means no save)
    fig_name=" "     # Figure name prefix (empty means default)
)

#Here these graphs below will show all of the SNR for each reduction you use. 
#Ex: if you ran reduction from Pc 1-8, all 8 CCF and SNR calculations would show below

# Evaluate the Significance of the Detection

Once the ccf is calculated, we can evaluate the significance a few different ways. 
1. T-test (statitstical test)
2. A "box" Method
3. Sigma Clipping Method.

Below we will walk through all three to compare how they quantify the same detection.

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Try to all three significance tests. Does anything change?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  Yes, the detection significance changes from method to method! 
</details>

# Generate the T-test

The t-test will be used to evaluate the null hypothesis that the two samples are drawn from the same underlying distribution. In this context, the test assesses whether any observed difference between the two datasets (such as the strength of a spectral feature or cross-correlation signal) is statistically significant or simply due to random noise. If the graph of in and out look the same, there is no clear detection.

In [ ]:
# === Generate the t-test map using the CCF object ===

# Parameters can be adjusted here to refine the map calculation
ccf_obj.ttest_map(
    all_obs[(n_pc, mask_wings)],      # Observation object for selected PCA and mask wings
    kind='logl',                      # Use log-likelihood method for t-test
    vrp=np.zeros_like(obs.vrp),       # Velocity residual profile (set to zero array here)
    orders=order_indices,             # Spectral orders to include in the analysis
    kp0=0,                           # Initial Kp value to start search (km/s)
    RV_limit=25,                    # Radial velocity limit (km/s)
    kp_step=5,                      # Step size for Kp grid (km/s)
    rv_step=1,                      # Step size for RV grid (km/s)
    RV=None,                        # Optional RV shift parameter
    speed_limit=3,                  # Speed limit (km/s) for smoothing or constraints
    equal_var=False                 # Assume unequal variance in t-test
)


# Box method:
1. We select a random region (a "box") away from the detection peak.
2. We compute the standard deviation (the sigma) of the values in this box.
3. We subtract the median from the Kp-Vsys map.
4. We divide the Kp-Vsys map by the standard deviation of the box. This gives us the SNR of the detection.

# Sigma clipping method:
This method is very similar to the box method, except we don’t use the standard deviation of a box-shaped region, but of a masked region that we find using a process called “sigma clipping”.
Sigma clipping is an iterative process. We start with the whole map. We calculate the median. Next, we find all values that deviate by more than X sigma from this median. We reject all these “extreme” values, and we move to the next iteration. We recompute the median, reject all extreme values, move to the new iteration, etc.
The region that is left after the sigma-clipping process is the region that we use to calculate the standard deviation.




In [ ]:
def calculate_KpVsys_map(OBS_OBJECT, CCF_OBJECT, method, rv_grid, kp_grid, box_Kp=None, box_Vrad=None, clip_sigma=2, clip_iter=4):
    

    """
    Compute and plot the Kp/Vrad cross-correlation map.

    This function generates a Kp/Vrad correlation map by summing the 
    cross-correlation functions (CCFs) along the planet radial velocity trail 
    as a function of Kp (semi-amplitude) and Vrad (systemic velocity shift). 
    It supports two normalization methods: 
    - 'box': standardizes using the standard deviation within a user-defined box.
    - 'clip': standardizes using sigma-clipping over the full map.

    The function produces a plot of the normalized Kp/Vrad map with annotations 
    for the maximum signal and reference lines.

    Parameters
    ----------
    OBS_OBJECT : Observation object (see planet_obs.py)
        Observation object containing the phases and relevant information.
    CCF_OBJECT : Correlations object (see correlation_class.py)
        CCF object containing the CCF map, RV grid, and Kp/Vrad arrays.
    method : str
        Normalization method, either 'box' or 'clip'. Default is 'box'.
    rv_grid : array-like
        Array of radial velocity offsets (Vrad) in km/s over which the map is computed. 
        Defines the x axis of the Kp/Vrad map.
    kp_grid : array-like
        Array of semi-amplitudes (Kp) in km/s over which the map is computed. 
        Defines the y axis of the Kp/Vrad map.
    box_Vrad : tuple of float, optional
        (min_Vrad, max_Vrad) limits in km/s for the box method. Required if method='box'.
    box_Kp : tuple of float, optional
        (min_Kp, max_Kp) limits in km/s for the box method. Required if method='box'.
    clip_sigma : float, optional
        Sigma threshold for sigma clipping when method='clip'. Default is 2.
    clip_iter : int, optional
        Maximum number of iterations for sigma clipping. Default is 4.

    Returns
    -------
    None
        Displays the Kp/Vrad map with annotations. No values are returned.

    Notes
    -----
    - The function assumes the CCF has been computed beforehand using cc.plot_ccflogl().
    - The box normalization requires specifying a Kp and Vsys range where 
      no signal is expected to compute the noise level.
    - In 'clip' mode, sigma-clipping is applied globally to remove outliers 
      before computing the noise standard deviation.
      
    Created by Joost Wardenier and improved by Mathis Bouffard, June 2025.
    
    """
    
    vsys_list = rv_grid
    Kp_list = kp_grid

    ######################################
    # 1. Build coordinate grid for plotting
    ######################################
    # Extend the coordinate arrays by one step for plotting with pcolormesh
    x_coords = np.concatenate((vsys_list, [vsys_list[-1]+np.diff(vsys_list)[-1]]))
    y_coords = np.concatenate((Kp_list, [Kp_list[-1]+np.diff(Kp_list)[-1]]))

    # Get spacings (deltas) between points
    delta_x = np.concatenate((np.diff(x_coords), [np.diff(x_coords)[-1]]))
    delta_y = np.concatenate((np.diff(y_coords), [np.diff(y_coords)[-1]]))

    # Center coordinates by half a delta
    x_coords = x_coords - 0.5 * delta_x
    y_coords = y_coords - 0.5 * delta_y

    # Create meshgrid for plotting
    XX_k, YY_k = np.meshgrid(x_coords, y_coords)

    ######################################
    # 2. Initialize Kp vs. Vsys map and variables
    ######################################
    RV_min = 1e6
    RV_max = 1e-6

    phases = OBS_OBJECT.phase           # The phase of observations
    velocities = CCF_OBJECT.rv_grid     # The radial velocity grid
    CCF = CCF_OBJECT.map_prf            # The cross-correlation function array

    KpVsys_map = np.zeros((len(Kp_list), len(vsys_list)))  # Output map

    phi_min = -0.02
    phi_max = 0.02
    mask = ((phases > phi_min) & (phases < phi_max))
    idx = np.where(mask == True)[0][0]

    selected_phases = phases[mask]

    ######################################
    # 3. Fill Kp vs. Vsys map
    ######################################
    for i, Kp in enumerate(Kp_list):
        for j, vsys in enumerate(vsys_list):
            CCF_sum = 0
            for k, norm_phase in enumerate(selected_phases):
                # Compute the planet RV shift
                RV = vsys + Kp * np.sin(2*np.pi*norm_phase)

                # Sum the CCF for this phase and shift
                CCF_sum = CCF_sum + np.interp(RV, velocities, CCF[idx + k, :], left=0., right=0.)

                # Update min and max RV
                if RV.max() > RV_max:
                    RV_max = RV.max()
                if RV.min() < RV_min:
                    RV_min = RV.min()

            # Store total summed CCF for this (Kp, vsys) point
            KpVsys_map[i, j] = CCF_sum

    # print('min RV =', RV_min, ' km/s | max RV = ', RV_max, ' km/s')

    ######################################
    # 4. Plotting the results
    ######################################
    fig = plt.figure(figsize=(10, 7))
    gs = gridspec.GridSpec(100, 42)

    ax1 = fig.add_subplot(gs[:96, 2:37])  # Main map
    axc = fig.add_subplot(gs[:96, 38:40]) # Colorbar

    # Choose normalization method
    if method == 'box':
        # Get index ranges for Kp
        kp_mask = (Kp_list >= np.min(box_Kp)) & (Kp_list <= np.max(box_Kp))
        kp_indices = np.where(kp_mask)[0]
        
        # Get index ranges for Vsys
        vsys_mask = (vsys_list >= np.min(box_Vrad)) & (vsys_list <= np.max(box_Vrad))
        vsys_indices = np.where(vsys_mask)[0]

        submap = KpVsys_map[np.ix_(kp_indices, vsys_indices)]  # the box
        
        sigma = np.std(submap)
        median = np.median(KpVsys_map)
        KpVsys_plot = (KpVsys_map - median) / sigma
        plot_title = 'BOX METHOD'
        
        # Draw rectangle outline for the selected box
        rect_x = [box_Vrad[0], box_Vrad[1], box_Vrad[1], box_Vrad[0], box_Vrad[0]]
        rect_y = [box_Kp[0], box_Kp[0], box_Kp[1], box_Kp[1], box_Kp[0]]
        ax1.plot(rect_x, rect_y, color='aqua', linestyle='dashdot', linewidth=1.5, zorder=10)
        
    else:
        # Sigma-clipping method
        masked_map = stats.sigma_clip(KpVsys_map, sigma=clip_sigma, maxiters=clip_iter)
        sigma_clip = np.std(masked_map)
        med_clip = np.median(KpVsys_map)
        KpVsys_plot = (KpVsys_map - med_clip) / sigma_clip
        plot_title = 'SIGMA-CLIPPING METHOD'

    # Main map
    ax1.pcolormesh(XX_k, YY_k, KpVsys_plot, cmap='inferno')
    max_idx = np.unravel_index(KpVsys_map.argmax(), KpVsys_map.shape)
    x_max = vsys_list[max_idx[1]]
    y_max = Kp_list[max_idx[0]]

    # Mark the central (0,Kp) position and best-fit point
    ax1.axvline(0, color='w', linewidth=2, linestyle='--')
    ax1.axhline(0, color='w', linewidth=2, linestyle='--')
    ax1.plot([x_max], [y_max], 'ko', zorder=10)

    # Labels and title
    ax1.set_title(plot_title, fontsize=25)
    ax1.set_xlabel('$\Delta \mathrm{v_{rad}}$ (km/s)', fontsize=25)
    ax1.set_ylabel('$\Delta \mathrm{K_{p}}$ (km/s)', fontsize=25)
    ax1.tick_params(labelsize=20)

    # Colorbar
    cmap = cm.inferno
    norm = Normalize(vmin=np.min(KpVsys_plot), vmax=np.max(KpVsys_plot))
    cb = ColorbarBase(axc, cmap=cmap, norm=norm, orientation='vertical')
    cb.set_label(label='SNR', fontsize=25)
    cb.ax.tick_params(labelsize=20)

    plt.tight_layout()
    plt.show()

<span style="font-size:24px; font-weight:bold; color:white; background-color:blue; display:block; padding:10px;">
Can you change the location of the box in the box method? Does this change the SNR? What happens if you change the clip sigma parameter in the sigma clipping method?
</span>

<details>
  <summary style="font-weight:bold; color:green; cursor:pointer;">✅ Click here to show the answer</summary>

  To change the box, change the box_Vrad and the box_kp numbers. This will change the SNR. Try changing the clip_sigma parameter and see the SNR change for the sigma clipping method
</details>

In [ ]:
# Choose Vrad (x-axis) Kp (y-axis) values for the plot
# rv_grid = corrRV0
rv_grid=np.linspace(-25,25,101)
kp_grid = np.linspace(-80, 80, 161)

# Set Vrad and Kp limits for the box in the box method
box_Vrad = (-5, 5) #vrad limits of the box
box_Kp = (0, 40) #kp limits of the box

# Set sigma limit and max number of iterations for the sigma-clipping method
clip_sigma = 2
clip_iter = 9

calculate_KpVsys_map(obs, ccf_obj, method='box', rv_grid=rv_grid, kp_grid=kp_grid,
                        box_Vrad=box_Vrad, box_Kp=box_Kp)
calculate_KpVsys_map(obs, ccf_obj, method='clip', rv_grid=rv_grid, kp_grid=kp_grid,
                        clip_sigma=clip_sigma, clip_iter=clip_iter)